# Cobolt Laser Blink Routine with Real‑Time Tracking
This Jupyter Notebook lets you set Cobolt laser power, blinking times, and **records the *actual* ON/OFF durations** measured on your computer’s clock.

> **Safety first ❗**  Make sure the interlock is closed, wear proper laser goggles, and double‑check the COM‑port before enabling emission.


## 1 – Core helper functions

In [10]:
import time
import datetime as dt
import serial
from contextlib import contextmanager
from typing import List, Tuple

@contextmanager
def cobolt_laser(port: str = "COM5", baud: int = 115200, timeout: float = 0.2):
    """Open Cobolt laser serial port and close it safely afterwards."""
    ser = serial.Serial(port, baudrate=baud, timeout=timeout)
    try:
        ser.write(b"\r")        # wake‑up
        time.sleep(0.05)
        ser.write(b"cf\r")      # clear faults
        yield ser
    finally:
        try:
            ser.write(b"!d\r")  # ensure emission off
        except Exception:
            pass
        ser.close()

def expect_ok(ser, sent_cmd: str):
    """Read reply to *sent_cmd*; raise with detail if not 'OK'."""
    reply = ser.readline().decode(errors="ignore").strip()
    print(f"<< {reply or '[no reply]'}")                # DEBUG
    if reply != "OK":
        raise RuntimeError(f"Cmd '{sent_cmd}' → '{reply}'")

def send_cmd(ser, cmd: str):
    """Send *cmd* terminated by CR and expect OK."""
    print(f">> {cmd}")                                  # DEBUG
    ser.write((cmd + "\r").encode())
    expect_ok(ser, cmd)

def set_power(ser, power_mw: int):
    send_cmd(ser, f"!p {power_mw}")

def laser_on(ser):
    send_cmd(ser, "!e")

def laser_off(ser):
    send_cmd(ser, "!d")

## 2 – Blink routine with real‑time tracking

In [11]:
def blink_sequence(
    port: str = "COM5",
    power_mw: int = 20,
    on_s: float = 1.0,
    off_s: float = 1.0,
    cycles: int = 10,
) -> Tuple[List[float], List[float]]:
    """Blink the laser and return two lists with actual ON and OFF durations (s)."""
    on_durations = []
    off_durations = []
    time_stamps = []  # optional timestamp log

    with cobolt_laser(port) as ser:
        set_power(ser, power_mw)

        for k in range(cycles):
            # --- Turn ON ---
            t_cmd_on_sent = time.perf_counter()
            laser_on(ser)
            t_on_start = time.perf_counter()
            time_stamps.append((dt.datetime.now(), 'ON'))
            # Planned ON dwell
            time.sleep(on_s)

            # --- Turn OFF ---
            t_cmd_off_sent = time.perf_counter()
            laser_off(ser)
            t_off_start = time.perf_counter()
            time_stamps.append((dt.datetime.now(), 'OFF'))

            on_durations.append(t_cmd_off_sent - t_on_start)  # actual ON time

            # Inter‑cycle OFF dwell (skip after last cycle)
            if k < cycles - 1:
                time.sleep(off_s)
                t_after_off_dwell = time.perf_counter()
                off_durations.append(t_after_off_dwell - t_off_start)

    # Return data
    return on_durations, off_durations


## 3 – Run the experiment & display statistics

In [12]:
# === Example: 0.5 s ON / 0.5 s OFF, 50 cycles at 50 mW ===
port = "COM8"         # change to your COM‑port
power_mw = 50
on_s = 0.5
off_s = 0.5
cycles = 50

on_times, off_times = blink_sequence(
    port=port,
    power_mw=power_mw,
    on_s=on_s,
    off_s=off_s,
    cycles=cycles,
)

print(f"Planned ON time per cycle: {on_s:.3f} s")
print(f"Average *actual* ON time : {sum(on_times)/len(on_times):.3f} s (min={min(on_times):.3f}, max={max(on_times):.3f})")
print(f"Planned OFF time per cycle: {off_s:.3f} s")
if off_times:
    print(f"Average *actual* OFF time: {sum(off_times)/len(off_times):.3f} s (min={min(off_times):.3f}, max={max(off_times):.3f})")


>> !p 50
<< Syntax error: illegal command


RuntimeError: Cmd '!p 50' → 'Syntax error: illegal command'

## 5 – Optional: Plot actual ON time per cycle

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(on_times, marker='o', linestyle='none')
plt.xlabel('Cycle #')
plt.ylabel('Actual ON time (s)')
plt.title('Cobolt ON‑time per cycle')
plt.grid(True)
plt.show()
